# BBO Capstone Project: Method and Results

This notebook presents the reproducible workflow for the Black-Box Optimisation capstone project. It is structured around the actual project objective: maximise unknown functions under a limited query budget.

The notebook expects a local query-response file at `data/raw/bbo_query_history.csv`. If that file is not present, it loads the shareable schema template and skips modelling cells without failing.

## 1. Project Setup

The workflow uses small-data Bayesian optimisation ideas. Query vectors are parsed from the challenge format, each function is modelled separately, and Gaussian Process surrogate models are used to rank candidate query points.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bbo_utils import (
    fit_gp_surrogate,
    load_query_history,
    recommend_candidates,
    summarise_best_by_function,
)

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)
PROJECT_ROOT

WindowsPath('C:/Users/ahmed/OneDrive/Desktop/Capstone-Project-ML-AI-Imperial-College')

## 2. Load the Query-Response History

The raw file should contain one row per submitted query with these columns: `function_id`, `round`, `query`, `output`, and optional `notes`. The query should be stored as a hyphen-separated vector, for example `0.123456-0.654321`.

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bbo_query_history.csv"
TEMPLATE_PATH = PROJECT_ROOT / "data" / "sample" / "bbo_query_history_template.csv"

selected_path = DATA_PATH if DATA_PATH.exists() else TEMPLATE_PATH
history = load_query_history(selected_path)

print(f"Loaded: {selected_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(history):,}")

if history.empty:
    print("The query history is empty. Add data/raw/bbo_query_history.csv to run modelling and results cells.")
else:
    display(history.head())

Loaded: data\sample\bbo_query_history_template.csv
Rows: 0
The query history is empty. Add data/raw/bbo_query_history.csv to run modelling and results cells.


## 3. Data Understanding

Review the number of observations per function, the dimensionality of each query vector, and the returned output values. Sparse histories and inconsistent outputs should be treated as uncertainty, not as evidence of a settled optimum.

In [3]:
if history.empty:
    print("Data understanding is skipped until query history is available.")
else:
    feature_columns = [column for column in history.columns if column.startswith("x")]
    summary = (
        history.groupby("function_id")
        .agg(
            observations=("output", "size"),
            first_round=("round", "min"),
            latest_round=("round", "max"),
            best_output=("output", "max"),
            mean_output=("output", "mean"),
        )
        .reset_index()
    )
    summary["query_dimensions"] = len(feature_columns)
    display(summary)
    display(summarise_best_by_function(history))

Data understanding is skipped until query history is available.


## 4. Exploratory Analysis

The most useful exploratory views for this project are optimisation-focused: best-so-far curves, output distributions, query clustering, boundary behaviour and signs of near-duplicate testing.

In [4]:
if history.empty:
    print("Exploratory plots are skipped until query history is available.")
else:
    plot_df = history.sort_values(["function_id", "round"]).copy()
    plot_df["best_so_far"] = plot_df.groupby("function_id")["output"].cummax()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.lineplot(data=plot_df, x="round", y="best_so_far", hue="function_id", marker="o", ax=ax)
    ax.set_title("Best Observed Output by Round")
    ax.set_xlabel("Round")
    ax.set_ylabel("Best output so far")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "best_so_far_by_round.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(data=history, x="function_id", y="output", ax=ax)
    ax.set_title("Returned Output Distribution by Function")
    ax.set_xlabel("Function")
    ax.set_ylabel("Output")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "output_distribution_by_function.png", dpi=300, bbox_inches="tight")
    plt.show()

Exploratory plots are skipped until query history is available.


## 5. Preprocessing for Surrogate Modelling

The helper functions validate query bounds and expand the query vector into numeric columns. Each black-box function should be modelled separately because the functions may have different dimensionality, smoothness, noise and local optima.

In [5]:
if history.empty:
    print("Preprocessing has no rows to process yet.")
else:
    feature_columns = sorted(
        [column for column in history.columns if column.startswith("x")],
        key=lambda column: int(column[1:]),
    )
    print("Feature columns:", feature_columns)
    display(history[["function_id", "round", "query", "output", *feature_columns]].head())

Preprocessing has no rows to process yet.


## 6. Modelling and Candidate Ranking

The primary model is a Gaussian Process surrogate. Candidate points can be ranked using Expected Improvement, which favours likely improvement over the current best, or Upper Confidence Bound, which gives explicit weight to uncertainty.

In [6]:
recommendation_tables = []

if history.empty:
    print("Surrogate modelling is skipped until query history is available.")
else:
    for function_id, function_df in history.groupby("function_id"):
        if len(function_df) < 2:
            print(f"Function {function_id}: skipped because at least two observations are required.")
            continue

        model, x_train, y_train = fit_gp_surrogate(function_df, random_state=RANDOM_STATE)
        recommendations = recommend_candidates(
            model,
            dimensions=x_train.shape[1],
            acquisition="ei",
            best_y=float(y_train.max()),
            n_candidates=5000,
            top_k=3,
            random_state=RANDOM_STATE,
        )
        recommendations.insert(0, "function_id", function_id)
        recommendation_tables.append(recommendations)

    if recommendation_tables:
        candidate_recommendations = pd.concat(recommendation_tables, ignore_index=True)
        candidate_recommendations.to_csv(REPORTS_DIR / "candidate_recommendations.csv", index=False)
        display(candidate_recommendations)
    else:
        print("No functions had enough observations for surrogate modelling.")

Surrogate modelling is skipped until query history is available.


## 7. Evaluation Metrics

This is an optimisation task, so the main metrics are not accuracy or F1 score. The relevant measures are best observed output, best-so-far improvement, query efficiency and whether the strategy avoids wasting limited queries.

In [7]:
if history.empty:
    print("Evaluation is skipped until query history is available.")
else:
    ordered = history.sort_values(["function_id", "round"]).copy()
    evaluation = (
        ordered.groupby("function_id")
        .agg(
            observations=("output", "size"),
            first_output=("output", "first"),
            best_output=("output", "max"),
            latest_round=("round", "max"),
        )
        .reset_index()
    )
    evaluation["absolute_improvement"] = evaluation["best_output"] - evaluation["first_output"]
    evaluation["improvement_per_query"] = evaluation["absolute_improvement"] / evaluation["observations"].clip(lower=1)
    evaluation.to_csv(REPORTS_DIR / "bbo_evaluation_summary.csv", index=False)
    display(evaluation)

Evaluation is skipped until query history is available.


## 8. Interpretation and Results

When the final query history is added, use this section to record the best result for each function, the strategy that produced it, and whether the final movement was mainly exploratory or exploitative.

Current repository status:

- Final numeric query history: not committed.
- Final score table: pending data.
- Main qualitative result: the strongest strategy became per-function, evidence-based and risk-controlled rather than one-size-fits-all.
- Most important limitation: the search space remains sparse, especially for higher-dimensional or noisy functions.

## 9. Responsible Use and Reproducibility

The workflow is designed for an educational capstone challenge. Model recommendations should be reviewed before use because Gaussian Process surrogates can be misleading when the data is sparse, noisy, discontinuous or concentrated around early promising regions.

For reproducibility, keep the query history, round number, model settings, acquisition function and any manual override notes together. A future reviewer should be able to understand why each query was selected, not only what the final query was.